In [4]:
!pip -q install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!ls

drive  kaggle.json  sample_data


In [7]:

from google.colab import drive
drive.mount('/content/drive')

# 6️⃣ Create folder in Drive
!mkdir -p /content/drive/MyDrive/imdb_data

# 7️⃣ Download dataset
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

# 8️⃣ Unzip directly into Drive folder
!unzip -q imdb-dataset-of-50k-movie-reviews.zip -d /content/drive/MyDrive/imdb_data

# 9️⃣ Check files
!ls /content/drive/MyDrive/imdb_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
  0% 0.00/25.7M [00:00<?, ?B/s]
100% 25.7M/25.7M [00:00<00:00, 913MB/s]
'IMDB Dataset.csv'


In [8]:
!ls /content/drive/MyDrive/imdb_data

'IMDB Dataset.csv'


In [9]:
!pip install pyspark

In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("IMDB Sentiment Analysis") \
        .getOrCreate()

spark

In [16]:
data = spark.read.csv(
    "/content/drive/MyDrive/imdb_data/IMDB Dataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

data.show(5, truncate=False)
data.printSchema()

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
from pyspark.sql.functions import when, col

data = data.withColumn(
    "label",
    when(col("sentiment") == "positive", 1).otherwise(0)
)

data.select("sentiment", "label").show(5)

+---------+-----+
|sentiment|label|
+---------+-----+
| positive|    1|
| positive|    1|
| positive|    1|
| negative|    0|
| positive|    1|
+---------+-----+
only showing top 5 rows


In [18]:
data.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1|25000|
|    0|25000|
+-----+-----+



IMDB Dataset
   ↓
Text Cleaning
   ↓
Tokenization
   ↓
FastText Embedding
   ↓
Sentence Vector (Average of word vectors)
   ↓
ANN Model
   ↓
Accuracy, Precision, Recall, F1 Score

In [20]:
from pyspark.sql.functions import col, lower, regexp_replace, trim

# lowercase
data = data.withColumn("review", lower(col("review")))

# remove html tags
data = data.withColumn("review", regexp_replace(col("review"), "<.*?>", ""))

# remove special characters and numbers
data = data.withColumn("review", regexp_replace(col("review"), "[^a-zA-Z\\s]", ""))

# remove extra spaces
data = data.withColumn("review", regexp_replace(col("review"), "\\s+", " "))

# trim spaces from start/end
data = data.withColumn("review", trim(col("review")))

data.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
from pyspark.ml.feature import Tokenizer
from pyspark.ml.clustering import KMeans

# Split data into training and testing sets
(train_data, test_data) = data.randomSplit([0.8, 0.2], seed=42)

tokenizer = Tokenizer(inputCol="review", outputCol="words")

train_data = tokenizer.transform(train_data)
test_data = tokenizer.transform(test_data)

train_data.select("review", "words").show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------

In [23]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647413 sha256=2e594fba6e0df3a239354237c1f79c1cb5b6818d8cf26fc1a4fdbffa9c7af8dd
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [24]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
!gunzip cc.en.300.bin.gz

--2026-03-02 19:07:36--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.171.22.118, 3.171.22.13, 3.171.22.68, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.171.22.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4503593528 (4.2G) [application/octet-stream]
Saving to: ‘cc.en.300.bin.gz’

cc.en.300.bin.gz    100%[===================>]   4.19G  92.7MB/s    in 51s     

2026-03-02 19:08:27 (83.9 MB/s) - ‘cc.en.300.bin.gz’ saved [4503593528/4503593528]



In [25]:
import fasttext

ft_model = fasttext.load_model("cc.en.300.bin")

In [26]:
print(ft_model.get_word_vector("good")[:10])

[-0.09213716 -0.0634383   0.00173813  0.13524324 -0.06561062  0.00619071
  0.12609869 -0.01646539  0.0174491  -0.00126792]


In [27]:
train_pd = train_data.select("words", "label").toPandas()
test_pd = test_data.select("words", "label").toPandas()

print(train_pd.shape)
print(test_pd.shape)

(39948, 2)
(10052, 2)


In [28]:
import numpy as np

def get_sentence_vector(words):
    vectors = []

    for word in words:
        try:
            vec = ft_model.get_word_vector(word)
            vectors.append(vec)
        except:
            continue

    if len(vectors) == 0:
        return np.zeros(300)
    else:
        return np.mean(vectors, axis=0)

# Train embeddings
X_train = np.vstack(train_pd["words"].apply(get_sentence_vector).values)
y_train = train_pd["label"].values

# Test embeddings
X_test = np.vstack(test_pd["words"].apply(get_sentence_vector).values)
y_test = test_pd["label"].values

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (39948, 300)
Test shape: (10052, 300)


In [29]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Build model
model = Sequential([
    Dense(128, activation='relu', input_shape=(300,)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        38,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,849 (183.00 KB)

 Trainable params: 46,849 (183.00 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/5
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7221 - loss: 0.5394 - val_accuracy: 0.8040 - val_loss: 0.4267
Epoch 2/5
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8249 - loss: 0.3954 - val_accuracy: 0.8340 - val_loss: 0.3832
Epoch 3/5
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8290 - loss: 0.3862 - val_accuracy: 0.8325 - val_loss: 0.3849
Epoch 4/5
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8373 - loss: 0.3722 - val_accuracy: 0.8343 - val_loss: 0.3747
Epoch 5/5
1124/1124 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.8354 - loss: 0.3740 - val_accuracy: 0.8368 - val_loss: 0.3747


In [31]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)

315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8315 - loss: 0.3738
Test Accuracy: 0.8333665132522583


In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predict probabilities
y_pred_probs = model.predict(X_test)

# Convert probabilities to 0/1
y_pred = (y_pred_probs > 0.5).astype(int)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print results
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

315/315 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Accuracy : 0.833366494230004
Precision: 0.8063277249451354
Recall   : 0.877412935323383
F1 Score : 0.8403697703230725

Confusion Matrix:
 [[3968 1059]
 [ 616 4409]]

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.79      0.83      5027
           1       0.81      0.88      0.84      5025

    accuracy                           0.83     10052
   macro avg       0.84      0.83      0.83     10052
weighted avg       0.84      0.83      0.83     10052

